# 00 — Infrastruktur hochfahren

## Zweck
Dieses Notebook startet die Infrastruktur, die das Projekt braucht: **Kafka** (Event-Broker),
**Spark** (Stream-Verarbeitung) und — lokal — **PostgreSQL** (Bronze-Speicher der EEA-Daten).
Es verarbeitet selbst noch keine Daten. Die fachlichen Schritte beginnen in Notebook `01`.

## Zwei Umgebungen
Welche Umgebung verwendet wird, steht allein in der `.env` (`EXECUTION_ENV`):

| `EXECUTION_ENV` | Bedeutung | Infrastruktur |
| --- | --- | --- |
| `docker_compose` | Lokaler Rechner mit Docker Desktop | `docker compose up` startet Kafka, Spark, PostgreSQL, Jupyter |
| `fh_jupyterhub` | FH-JupyterHub | Kafka/Spark laufen bereits auf der FH; hier nur Erreichbarkeit prüfen |

Vorlage kopieren: `.env.example` (lokal) bzw. `.env.cluster.example` (FH) nach `.env`.

## Projektpfad und Umgebung bestimmen

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)

EXECUTION_ENV = os.getenv("EXECUTION_ENV", "docker_compose")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")

print({"project_root": str(PROJECT_ROOT), "execution_env": EXECUTION_ENV})

## Hilfsfunktion: warten, bis ein Port erreichbar ist

In [ ]:
import socket
import time


def wait_for_port(host: str, port: int, timeout_seconds: int = 180) -> bool:
    deadline = time.monotonic() + timeout_seconds
    while time.monotonic() < deadline:
        try:
            with socket.create_connection((host, port), timeout=3):
                return True
        except OSError:
            time.sleep(3)
    return False

## Infrastruktur starten

**Docker:** `docker compose up -d --build` baut beim ersten Mal die Images (einige Minuten)
und startet danach alle Dienste. Anschließend warten wir, bis die Ports erreichbar sind.

**FH:** kein Docker — wir prüfen nur, ob der FH-Kafka-Broker erreichbar ist.

In [ ]:
import subprocess

if EXECUTION_ENV == "docker_compose":
    # Setzt voraus, dass Docker Desktop laeuft.
    subprocess.run(["docker", "compose", "up", "-d", "--build"], cwd=PROJECT_ROOT, check=True)

    ports = {"kafka": 9092, "spark_master": 7077, "postgres": 5432, "jupyter": 8888}
    status = {name: wait_for_port("localhost", port) for name, port in ports.items()}
    assert all(status.values()), f"Nicht alle Dienste sind erreichbar: {status}"
    print({"infrastruktur": "docker_compose", "ports_erreichbar": status})
    print("Naechster Schritt: http://localhost:8888 oeffnen und Notebooks 01-10 dort ausfuehren.")
else:
    host, port = KAFKA_BOOTSTRAP_SERVERS.rsplit(":", 1)
    reachable = wait_for_port(host, int(port), timeout_seconds=15)
    assert reachable, f"FH-Kafka {KAFKA_BOOTSTRAP_SERVERS} ist nicht erreichbar."
    print({"infrastruktur": "fh_jupyterhub", "kafka": KAFKA_BOOTSTRAP_SERVERS, "erreichbar": reachable})
    print("Naechster Schritt: Notebooks 01-10 im FH-JupyterHub ausfuehren.")

## Nächster Schritt
Notebook `01` öffnen — dort werden Projektumfang, Leitfrage und Anforderungen beschrieben.